Регрессия: предсказание CC50
Аналогичная методология: сравнение и настройка гиперпараметров моделей.

In [1]:
import os
os.makedirs("plots", exist_ok=True)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.feature_selection import VarianceThreshold
import warnings

warnings.filterwarnings("ignore")

## Данные

In [2]:
df = pd.read_excel("data/data.xlsx", index_col=0)
targets = ["IC50, mM", "CC50, mM", "SI"]
features = [c for c in df.columns if c not in targets]

X = df[features]
y = np.log1p(df["CC50, mM"])

selector = VarianceThreshold(threshold=0.0)
X = X.fillna(X.median())
X_sel = selector.fit_transform(X)
sel_features = [features[i] for i in selector.get_support(indices=True)]
X = pd.DataFrame(X_sel, columns=sel_features)
print(f"Признаков после фильтрации: {X.shape[1]}")

cv = KFold(n_splits=5, shuffle=True, random_state=42)


def cv_rmse(model, X, y, cv):
    return -cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error").mean()

Признаков после фильтрации: 192


## Базовые модели

In [3]:
print("\n── Базовые модели (дефолтные параметры) ──")
base_models = {
    "Ridge":            Pipeline([("sc", StandardScaler()), ("m", Ridge())]),
    "Lasso":            Pipeline([("sc", StandardScaler()), ("m", Lasso(max_iter=5000))]),
    "ElasticNet":       Pipeline([("sc", StandardScaler()), ("m", ElasticNet(max_iter=5000))]),
    "RandomForest":     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}
for name, model in base_models.items():
    rmse = cv_rmse(model, X, y, cv)
    r2 = cross_val_score(model, X, y, cv=cv, scoring="r2").mean()
    print(f"  {name:22s}: RMSE={rmse:.4f}  R²={r2:.4f}")


── Базовые модели (дефолтные параметры) ──
  Ridge                 : RMSE=2.1966  R²=-1.5740
  Lasso                 : RMSE=1.5867  R²=-0.0026
  ElasticNet            : RMSE=1.5867  R²=-0.0026
  RandomForest          : RMSE=1.2043  R²=0.4211
  GradientBoosting      : RMSE=1.2115  R²=0.4141


## GridSearchCV

In [4]:
print("\n── GridSearchCV: Ridge ──")
gs_ridge = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", Ridge())]),
    {"m__alpha": [0.01, 0.1, 1, 10, 100, 500]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_ridge.fit(X, y)
print(f"  alpha={gs_ridge.best_params_}  RMSE={-gs_ridge.best_score_:.4f}")

print("\n── GridSearchCV: Lasso ──")
gs_lasso = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", Lasso(max_iter=10000))]),
    {"m__alpha": [0.001, 0.01, 0.1, 0.5, 1]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_lasso.fit(X, y)
print(f"  alpha={gs_lasso.best_params_}  RMSE={-gs_lasso.best_score_:.4f}")

print("\n── GridSearchCV: RandomForest ──")
gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_rf.fit(X, y)
print(f"  params={gs_rf.best_params_}  RMSE={-gs_rf.best_score_:.4f}")

print("\n── GridSearchCV: GradientBoosting ──")
gs_gb = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1, 0.2], "max_depth": [3, 5]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_gb.fit(X, y)
print(f"  params={gs_gb.best_params_}  RMSE={-gs_gb.best_score_:.4f}")


── GridSearchCV: Ridge ──
  alpha={'m__alpha': 500}  RMSE=1.7153

── GridSearchCV: Lasso ──
  alpha={'m__alpha': 0.1}  RMSE=1.3672

── GridSearchCV: RandomForest ──
  params={'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}  RMSE=1.1985

── GridSearchCV: GradientBoosting ──
  params={'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}  RMSE=1.2106


## Итог

In [5]:
print("\n── Итоговое сравнение RMSE (tuned) ──")
tuned = {
    "Ridge":            -gs_ridge.best_score_,
    "Lasso":            -gs_lasso.best_score_,
    "RandomForest":     -gs_rf.best_score_,
    "GradientBoosting": -gs_gb.best_score_,
}
for name, rmse in sorted(tuned.items(), key=lambda x: x[1]):
    print(f"  {name:22s}: RMSE = {rmse:.4f}")


── Итоговое сравнение RMSE (tuned) ──
  RandomForest          : RMSE = 1.1985
  GradientBoosting      : RMSE = 1.2106
  Lasso                 : RMSE = 1.3672
  Ridge                 : RMSE = 1.7153


## Feature Importance

In [6]:
importances = gs_rf.best_estimator_.feature_importances_
top_idx = np.argsort(importances)[-20:]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([X.columns[i] for i in top_idx], importances[top_idx], color="coral")
ax.set_title("RandomForest: Топ-20 важных признаков (CC50)")
ax.set_xlabel("Feature Importance")
plt.tight_layout()
plt.savefig("plots/cc50_feature_importance.png")
plt.close()
print("\nСохранён график: cc50_feature_importance.png")


Сохранён график: cc50_feature_importance.png


## Предсказания vs Факт

In [ ]:
y_pred = gs_gb.predict(X)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y, y_pred, alpha=0.3, s=10, color="coral")
mn, mx = y.min(), y.max()
ax.plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Идеальное предсказание")
ax.set_xlabel("log1p(CC50) — факт")
ax.set_ylabel("log1p(CC50) — предсказание")
ax.set_title(f"GradientBoosting (tuned): R²={r2_score(y, y_pred):.3f}")
ax.legend()
plt.tight_layout()
plt.savefig("plots/cc50_pred_vs_actual.png")
plt.close()

## Итоговые выводы

- После фильтрации осталось 192 признака.
- Линейные модели показывают очень низкое качество: Ridge имеет сильно отрицательный R² (-1.57), Lasso и ElasticNet близки к случайному угадыванию (R² около нуля).
- Random Forest достигает R²=0.42, GradientBoosting — R²=0.41.
- Настройка гиперпараметров улучшила Lasso (RMSE снизился с 1.587 до 1.367), но он всё равно заметно уступает деревьям.
- Лучшая модель — Random Forest с параметрами max_depth=None, min_samples_split=5, n_estimators=200 (RMSE=1.1985).
- GradientBoosting после настройки даёт близкий результат (RMSE=1.2106).
- Разрыв между лучшей линейной моделью (Lasso tuned, RMSE=1.367) и лучшей древовидной (Random Forest, RMSE=1.199) составляет 0.17 — это подтверждает наличие нелинейных зависимостей.
- Качество предсказания остаётся умеренным (R²≈0.42) — селективность предсказывать трудно.